# Kaggle Test Set Prediction

**Objective:** Load the labelled training data prepared in Notebook 2, apply the same classification preprocessing to Kaggle's unlabelled test data, train the selected TF-IDF model on all labelled rows, and create `submission.csv`.

The Kaggle test set is not part of `train_dataset_preprocessed.pkl`; this notebook loads it separately from `Datasets/test.csv`. Test IDs are preserved and aligned to `sample_submission.csv` when writing the output.

## 1. Load the prepared training data and Kaggle test data

Notebook 2 saved the processed, labelled training rows. The Kaggle test CSV contains the test lyrics and IDs, but no genre labels.

In [21]:
import pickle
import re

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import SGDClassifier

from utils import processing_pipeline_for_classification

with open("train_dataset_preprocessed.pkl", "rb") as f:
    train_dataset = pickle.load(f)

test_dataset = pd.read_csv("Datasets/test.csv", index_col="id")

assert "lyrics_processed_for_classification" in train_dataset.columns
assert "tag" in train_dataset.columns
assert "lyrics" in test_dataset.columns

print(f"Labelled training rows: {len(train_dataset):,}")
print(f"Kaggle test rows: {len(test_dataset):,}")

Labelled training rows: 133,034
Kaggle test rows: 33,742


## 2. Apply the classification preprocessing to Kaggle test lyrics

Use the same regex rules as Notebook 2, then the shared tokenization, stopword-removal, and lemmatization pipeline. Do not remove any test rows: Kaggle expects one prediction for every test ID.

In [22]:
test_dataset

,title,artist,year,views,features,lyrics
id,,,,,,
3996359,Like You Do,Jimmie Allen,2018,825,{},[Verse 1]\r\nYour head's on my chest\r\nThis b...
3553166,This Building is Falling Down,The Taxpayers,2009,400,{},Epilogue: A sound off from simple people with ...
2998742,My Private Joy,Dion and The Belmonts,1960,229,{},[Verse 1]\r\nOh well Susie is a doozy with a p...
3993470,Vegan,Yung Kurry,2018,1041,{},Hook\r\nShe a vegan but she eat my meat with n...
5488256,To The Boy I Used To Know,Angelina Cruz,2018,63,{},To the boy I used to know\r\nI wonder how do y...
...,...,...,...,...,...,...
1149534,My Life With You,Celtic Thunder,2015,32,{},When I was a younger man\r\nWhen I was a silly...
2473358,Work it out,The Jacka,2014,488,{Husalah},[Intro]\r\nNigga did you take my money?\r\nHeh...
5254514,No More,Darling West,2016,20,{},The moon is hanging high above\r\nA long and d...


In [23]:
regex_emojis = re.compile(r"[\u2600-\u27FF]|[\U00010000-\U0010ffff]")
regex_hashtags = re.compile(r"#\S+")
regex_mentions = re.compile(r"@\S+")
regex_urls = re.compile(r"https?://\S+|www\.\S+")
regex_tags = re.compile(r"\[|\]|\(|\)")
regex_standalone_words = re.compile(
    r"(?i)\b(?:verse|chorus|hook|pre|bridge|outro|intro|post|refrain|instrumental)\s*\d*\b"
)
regex_newlines = re.compile(r"[\n\t\r]+")
regex_apostrophes = re.compile(r"['’]")
regex_punctuation = re.compile(r"[^a-zA-Z0-9\s]")
regex_extra_spaces = re.compile(r"\s{2,}")


def regex_clean_lyrics_for_classification(text):
    text = regex_urls.sub(" ", text)
    text = regex_mentions.sub(" ", text)
    text = regex_hashtags.sub(" ", text)
    #text = regex_tags.sub(" ", text)
    #text = regex_standalone_words.sub(" ", text)
    text = regex_emojis.sub(" ", text)
    text = regex_newlines.sub(" ", text)
    text = regex_apostrophes.sub("", text)
    text = regex_punctuation.sub(" ", text)
    text = text.lower()
    text = regex_extra_spaces.sub(" ", text)
    return text.strip()

test_dataset["lyrics_regex_cleaned_for_classification"] = (
    test_dataset["lyrics"].fillna("").astype(str).map(regex_clean_lyrics_for_classification)
)

test_dataset["lyrics_processed_for_classification"] = (
    test_dataset["lyrics_regex_cleaned_for_classification"].map(
        lambda text: processing_pipeline_for_classification.tokenization_processing(
            text,
            print_output=False,
        )
    )
)

print("Processed test rows:", len(test_dataset))
print(test_dataset["lyrics_processed_for_classification"].head(2))

Processed test rows: 33742
id
3996359    verse 1 head chest bed mess empty bottle red w...
3553166    epilogue sound simple people simple theory fuc...
Name: lyrics_processed_for_classification, dtype: object


## 3. Fit TF-IDF and the selected classifier on all labelled data

Validation selected the combined word + character TF-IDF features with SGD Logistic (`alpha=0.0001`, `class_weight="balanced"`). Fit both TF-IDF vectorizers and the classifier on all labelled rows, then transform and predict the Kaggle test rows. The test data is never used for fitting.

In [24]:
from sklearn.pipeline import FeatureUnion

X_train_full = train_dataset["lyrics_processed_for_classification"].fillna("").astype(str)
y_train_full = train_dataset["tag"]
X_test = test_dataset["lyrics_processed_for_classification"].fillna("").astype(str)

tfidf_vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w+\b",
        min_df=3,
        max_features=20000,
    )),
    ("char", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=3,
        max_features=100000,
    )),
])

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_full)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

final_model = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=2000,
    random_state=42,
    alpha=0.0001,
)
final_model.fit(X_train_tfidf, y_train_full)
test_predictions = final_model.predict(X_test_tfidf)

print("Predictions created:", len(test_predictions))

Predictions created: 33742


## 4. Create and validate the Kaggle submission

Use the IDs and column order from `sample_submission.csv` so the output matches Kaggle's expected format.

In [25]:
sample_submission = pd.read_csv("Datasets/sample_submission.csv")

predictions_by_id = pd.Series(test_predictions, index=test_dataset.index, name="tag")
submission = (
    predictions_by_id.reindex(sample_submission["id"])
    .rename_axis("id")
    .reset_index()
)

assert len(submission) == len(sample_submission)
assert submission["tag"].notna().all()
assert submission.columns.tolist() == ["id", "tag"]
assert submission["id"].equals(sample_submission["id"])

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())

Saved submission.csv


,id,tag
0,3230388,misc
1,3658643,pop
2,1270714,pop
3,3354312,rap
4,493974,pop
